In [ ]:
import os
import numpy as np
import torch
import pandas as pd
from tqdm import tqdm
from datasets import Dataset
import evaluate
import nltk
import matplotlib.pyplot as plt
nltk.download("wordnet", quiet=True)

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    set_seed
)

# ======================
# CONFIG
# ======================
MODEL_NAME = "Osame1/AraT5-msa-small-finetuned-xlsum-ar"
DATA_FILE  = "/kaggle/input/versionfinal2/merged_datasetfinalv2 (2).csv"
OUTPUT_DIR = "/kaggle/working/t5-arabic-without_finetuning"

MAX_INPUT_LENGTH  = 384
MAX_TARGET_LENGTH = 96
BATCH_SIZE = 8  # Larger batch for inference speed
SEED = 42

set_seed(SEED)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ======================
# LOAD DATA
# ======================
def load_csv(path):
    if not os.path.exists(path):
        raise FileNotFoundError(f"Error: The data file '{path}' was not found.")

    df = pd.read_csv(path)
    data = {"text": [], "summary": []}
    for _, r in tqdm(df.iterrows(), total=len(df), desc="Loading data"):
        t, s = str(r["text"]).strip(), str(r["summary"]).strip()
        if t and s:
            data["text"].append(t)
            data["summary"].append(s)
    return Dataset.from_dict(data)

dataset = load_csv(DATA_FILE)

# Create validation split (10%)
dataset = dataset.train_test_split(test_size=0.1, seed=SEED)
eval_ds = dataset["test"]
print(f"✅ Loaded {len(eval_ds)} samples for evaluation")

# ======================
# LOAD MODEL & TOKENIZER
# ======================
print(f"📥 Loading model: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
model.eval()  # Set to evaluation mode
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
print(f"✅ Model loaded on {device}")

# ======================
# INFERENCE: Generate predictions in batches
# ======================
def generate_summaries(texts):
    """Generate summaries for a batch of texts"""
    inputs = tokenizer(
        [f"النص: {t}\nالملخص:" for t in texts],
        max_length=MAX_INPUT_LENGTH,
        truncation=True,
        padding="max_length",
        return_tensors="pt"
    ).to(device)
    
    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_length=MAX_TARGET_LENGTH,
            min_length=10,
            num_beams=4,
            early_stopping=True,
            no_repeat_ngram_size=2,
            use_cache=False  # 🔑 CRITICAL FIX
        )
    
    return tokenizer.batch_decode(outputs, skip_special_tokens=True)

# Process entire dataset in batches
all_predictions = []
all_references = []

print("\n🤖 Generating summaries...")
for i in tqdm(range(0, len(eval_ds), BATCH_SIZE), desc="Batches"):
    batch_texts = eval_ds[i:i+BATCH_SIZE]["text"]
    batch_summaries = eval_ds[i:i+BATCH_SIZE]["summary"]
    
    predictions = generate_summaries(batch_texts)
    
    all_predictions.extend(predictions)
    all_references.extend(batch_summaries)

print(f"✅ Generated {len(all_predictions)} summaries")

# ======================
# EVALUATION METRICS
# ======================
print("\n📊 Computing metrics...")

bertscore_metric = evaluate.load("bertscore")
chrf_metric      = evaluate.load("chrf")
bleu_metric      = evaluate.load("bleu")
meteor_metric    = evaluate.load("meteor")

# BERTScore (Arabic)
bertscore = bertscore_metric.compute(
    predictions=all_predictions,
    references=all_references,
    lang="ar"
)

# chrF
chrf = chrf_metric.compute(
    predictions=all_predictions,
    references=all_references
)

# BLEU
bleu = bleu_metric.compute(
    predictions=all_predictions,
    references=[[ref] for ref in all_references]
)

# METEOR
meteor = meteor_metric.compute(
    predictions=all_predictions,
    references=all_references
)

metrics = {
    "bertscore_precision": np.mean(bertscore["precision"]),
    "bertscore_recall":    np.mean(bertscore["recall"]),
    "bertscore_f1":        np.mean(bertscore["f1"]),
    "chrf": chrf["score"],
    "bleu": bleu["bleu"],
    "meteor": meteor["meteor"],
}

# ======================
# RESULTS REPORT
# ======================
print("\n" + "="*60)
print("EVALUATION RESULTS (Zero-Shot Inference)")
print("="*60)
print(f"Model: {MODEL_NAME}")
print(f"Samples evaluated: {len(eval_ds)}")
print("-"*60)
for k, v in sorted(metrics.items()):
    if isinstance(v, float):
        print(f"{k:25s}: {v:.4f}")
print("="*60)

# Save metrics to JSON
import json
with open(os.path.join(OUTPUT_DIR, "metrics.json"), "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2, ensure_ascii=False)
print(f"\n✅ Metrics saved to: {OUTPUT_DIR}/metrics.json")

# ======================
# SAVE PREDICTIONS FOR ANALYSIS
# ======================
results_df = pd.DataFrame({
    "text": eval_ds["text"],
    "reference_summary": eval_ds["summary"],
    "generated_summary": all_predictions
})

# Add BERTScore per sample for analysis
results_df["bertscore_f1"] = bertscore["f1"]

# Save full results
results_path = os.path.join(OUTPUT_DIR, "predictions.csv")
results_df.to_csv(results_path, index=False, encoding="utf-8-sig")
print(f"✅ Predictions saved to: {results_path}")

# Save sample outputs for quick inspection
sample_path = os.path.join(OUTPUT_DIR, "sample_outputs.txt")
with open(sample_path, "w", encoding="utf-8") as f:
    f.write("="*70 + "\n")
    f.write("SAMPLE SUMMARIZATION RESULTS (First 10 examples)\n")
    f.write("="*70 + "\n\n")
    for i in range(min(10, len(results_df))):
        f.write(f"TEXT {i+1}:\n{results_df.iloc[i]['text'][:300]}...\n\n")
        f.write(f"REFERENCE:\n{results_df.iloc[i]['reference_summary']}\n\n")
        f.write(f"GENERATED:\n{results_df.iloc[i]['generated_summary']}\n\n")
        f.write(f"BERTScore F1: {results_df.iloc[i]['bertscore_f1']:.4f}\n")
        f.write("-"*70 + "\n\n")
print(f"✅ Sample outputs saved to: {sample_path}")

# ======================
# VISUALIZATIONS
# ======================
plots_dir = os.path.join(OUTPUT_DIR, "plots")
os.makedirs(plots_dir, exist_ok=True)

# 1. Metric Scores Bar Chart
plt.figure(figsize=(10, 6))
metric_names = ["BERTScore F1", "chrF", "BLEU", "METEOR"]
metric_values = [
    metrics["bertscore_f1"],
    metrics["chrf"] / 100,  # Normalize chrF (0-100) to 0-1 for fair comparison
    metrics["bleu"] / 100,  # Normalize BLEU (0-100) to 0-1
    metrics["meteor"]
]
colors = ['#2E86AB', '#A23B72', '#F18F01', '#C73E1D']

bars = plt.bar(metric_names, metric_values, color=colors, alpha=0.85)
plt.ylabel('Score', fontsize=12)
plt.title('Summary Quality Metrics (Zero-Shot)', fontsize=14, fontweight='bold')
plt.ylim(0, 1)
plt.grid(axis='y', alpha=0.3)

# Add value labels on bars
for bar, val in zip(bars, metric_values):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
             f'{val:.3f}', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(plots_dir, "metrics_comparison.png"), dpi=150)
plt.close()
print(f"✅ Metrics plot saved to: {plots_dir}/metrics_comparison.png")

# 2. BERTScore Distribution
plt.figure(figsize=(8, 5))
plt.hist(bertscore["f1"], bins=30, color='#2E86AB', edgecolor='white', alpha=0.85)
plt.axvline(np.mean(bertscore["f1"]), color='red', linestyle='--', 
            label=f'Mean: {np.mean(bertscore["f1"]):.3f}')
plt.xlabel('BERTScore F1', fontsize=11)
plt.ylabel('Frequency', fontsize=11)
plt.title('Distribution of BERTScore F1 (Per Sample)', fontsize=13, fontweight='bold')
plt.legend()
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(plots_dir, "bertscore_distribution.png"), dpi=150)
plt.close()
print(f"✅ BERTScore distribution plot saved to: {plots_dir}/bertscore_distribution.png")

# 3. Summary Length Analysis
ref_lengths = [len(s.split()) for s in all_references]
pred_lengths = [len(s.split()) for s in all_predictions]

plt.figure(figsize=(8, 5))
plt.hist(ref_lengths, bins=30, alpha=0.7, label='Reference', color='#2E86AB')
plt.hist(pred_lengths, bins=30, alpha=0.7, label='Generated', color='#A23B72')
plt.axvline(np.mean(ref_lengths), color='#2E86AB', linestyle='--', 
            label=f'Ref Mean: {np.mean(ref_lengths):.1f}')
plt.axvline(np.mean(pred_lengths), color='#A23B72', linestyle='--', 
            label=f'Gen Mean: {np.mean(pred_lengths):.1f}')
plt.xlabel('Summary Length (words)', fontsize=11)
plt.ylabel('Frequency', fontsize=11)
plt.title('Summary Length Distribution', fontsize=13, fontweight='bold')
plt.legend()
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(plots_dir, "length_distribution.png"), dpi=150)
plt.close()
print(f"✅ Length distribution plot saved to: {plots_dir}/length_distribution.png")

print("\n" + "="*60)
print("✅ INFERENCE COMPLETE")
print("="*60)
print(f"Results directory: {OUTPUT_DIR}")
print(f" - metrics.json          : Quantitative results")
print(f" - predictions.csv       : Full predictions + references")
print(f" - sample_outputs.txt    : Human-readable samples")
print(f" - plots/                : Visualizations")
print("="*60)